# Creating Null Models for Novel Domain Architectures

Given the results from the cancer fusion gene analysis, we will establish some null models to determine some of the effect sizes and false discoveries we may be observing in that data.

In [1]:
import pandas as pd
import numpy as np
import dansy
import random
import itertools
from collections import Counter
import fusionClasses as fc
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import scipy.stats as stats
import networkx as nx
import EnrichementAnalysis

In [2]:
ref_df = dansy.import_proteome_files(ref_file_dir='./data/Current_Human_Proteome',
                                     ref_file_suffix='2026_0324.csv')
exon_information = pd.read_csv('Gene_exon_information.csv', index_col=0)
gene_conv = pd.read_csv('ENSEMBL_Gene_Conversion.csv')
valid_uniprots = list(set(ref_df['UniProt ID']).intersection(gene_conv['UniProtKB/Swiss-Prot ID'].unique()))

In [3]:
proteome_net = dansy.dansy(ref=ref_df, n=10)

Starting to fetch n-grams.
Finished getting all n-grams
Starting to generate adjacency
Finished building adjacency.


In [4]:
x = gene_conv.filter(['Gene stable ID', 'Gene name','UniProtKB/Swiss-Prot ID','Chromosome/scaffold name']).drop_duplicates()
conv_dict = x.set_index('UniProtKB/Swiss-Prot ID')['Gene stable ID'].to_dict()
name_conv = x.set_index('UniProtKB/Swiss-Prot ID')['Gene name'].to_dict()
chr_info = x.set_index('UniProtKB/Swiss-Prot ID')['Chromosome/scaffold name'].to_dict()

In [5]:
# Now let's take all the pairs and within each randomly choose exons to include
exons_grouped = exon_information.groupby('Gene stable ID')

In [6]:
# run through each pair and grab a random exon start point to call the breakpoint
natural_ngrams  = set(proteome_net.ngrams).union(proteome_net.collapsed_ngrams)
null_res_list = []
for i in range(100):
    random.seed(i*2)
    n = 15000 # How many pairs to create
    pairs = np.reshape(random.choices(valid_uniprots, k=n*2), (n,2)).tolist()
    null_fusions = []
    for pair in pairs:
        bp_info = pd.Series(index=['h_uniprot', 't_uniprot','h_pos','t_pos', 'h_chr', 't_chr', 'h_gene', 't_gene', 'h_symbol','t_symbol'],
                            dtype=object)
        bp_info['h_uniprot'] = pair[0]
        bp_info['t_uniprot'] = pair[1]
        bp_info['h_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[0]])['Exon region start (bp)'].values)
        bp_info['t_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[1]])['Exon region start (bp)'].values)
        bp_info['h_gene'] = conv_dict[pair[0]]
        bp_info['t_gene'] = conv_dict[pair[1]]
        bp_info['h_symbol'] = name_conv[pair[0]]
        bp_info['t_symbol'] = name_conv[pair[1]]
        bp_info['h_chr'] = chr_info[pair[0]]
        bp_info['t_chr'] = chr_info[pair[1]]
        null_fusions.append(bp_info)
        
    null_fusions = pd.DataFrame.from_records(null_fusions)
    null_fusions['name'] = null_fusions.h_symbol.str.cat(null_fusions.t_symbol, sep = '--')
    fusion_dansy = fc.fusionCollection(null_fusions, mapper={k:k for k in null_fusions.columns}, add_cols=[])
    fusion_dansy.perform_dansy_analysis(dansy_obj=proteome_net)
    y = fusion_dansy.summarize_dansy_results()
    y['Novel_Architecture'] = y['dansy_impact'].apply(lambda x: str('Novel' in x))
    # Let's grab all the n-grams and all the domains from the natural proteome
    null_ngrams = [k for k in dansy.ngramUtilities.return_ngrams_from_list(y.domain_architecture.values,3)]
    
    # Now limiting them to only those found in the proteome
    null_ngrams = list(set(null_ngrams).intersection(natural_ngrams))
    null_enrichment_novel = EnrichementAnalysis.EnrichmentAnalysis(y,null_ngrams, {'category':'Novel_Architecture','domain_architecture':'domain_architecture'})
    null_res_list.append(null_enrichment_novel.res.set_index('ngram').True_q)

100%|██████████| 6345/6345 [00:14<00:00, 451.17it/s]


There were 2155 fusion domain architectures previously found in the proteome.


100%|██████████| 6402/6402 [00:16<00:00, 396.16it/s]


There were 2093 fusion domain architectures previously found in the proteome.


100%|██████████| 6435/6435 [00:14<00:00, 445.05it/s]


...

100%|██████████| 6462/6462 [00:15<00:00, 421.86it/s]


There were 2132 fusion domain architectures previously found in the proteome.


100%|██████████| 6514/6514 [00:16<00:00, 396.74it/s]


There were 2205 fusion domain architectures previously found in the proteome.


100%|██████████| 9056/9056 [00:34<00:00, 264.51it/s]


In [7]:
tcga_null_dists = pd.concat(null_res_list, axis = 1)
tcga_null_dists.to_csv('Null_p_val_dists_15K_tcga.csv')

Now repeating but with fewer pairs to recapture the cutoffs for the CCLE datasets instead

In [8]:
# run through each pair and grab a random exon start point to call the breakpoint
natural_ngrams  = set(proteome_net.ngrams).union(proteome_net.collapsed_ngrams)
null_res_list = []
for i in range(100):
    random.seed(i*2)
    n = 6500 # How many pairs to create
    pairs = np.reshape(random.choices(valid_uniprots, k=n*2), (n,2)).tolist()
    null_fusions = []
    for pair in pairs:
        bp_info = pd.Series(index=['h_uniprot', 't_uniprot','h_pos','t_pos', 'h_chr', 't_chr', 'h_gene', 't_gene', 'h_symbol','t_symbol'],
                            dtype=object)
        bp_info['h_uniprot'] = pair[0]
        bp_info['t_uniprot'] = pair[1]
        bp_info['h_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[0]])['Exon region start (bp)'].values)
        bp_info['t_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[1]])['Exon region start (bp)'].values)
        bp_info['h_gene'] = conv_dict[pair[0]]
        bp_info['t_gene'] = conv_dict[pair[1]]
        bp_info['h_symbol'] = name_conv[pair[0]]
        bp_info['t_symbol'] = name_conv[pair[1]]
        bp_info['h_chr'] = chr_info[pair[0]]
        bp_info['t_chr'] = chr_info[pair[1]]
        null_fusions.append(bp_info)
        
    null_fusions = pd.DataFrame.from_records(null_fusions)
    null_fusions['name'] = null_fusions.h_symbol.str.cat(null_fusions.t_symbol, sep = '--')
    fusion_dansy = fc.fusionCollection(null_fusions, mapper={k:k for k in null_fusions.columns}, add_cols=[])
    fusion_dansy.perform_dansy_analysis(dansy_obj=proteome_net)
    y = fusion_dansy.summarize_dansy_results()
    y['Novel_Architecture'] = y['dansy_impact'].apply(lambda x: str('Novel' in x))
    # Let's grab all the n-grams and all the domains from the natural proteome
    null_ngrams = [k for k in dansy.ngramUtilities.return_ngrams_from_list(y.domain_architecture.values,3)]
    
    # Now limiting them to only those found in the proteome
    null_ngrams = list(set(null_ngrams).intersection(natural_ngrams))
    null_enrichment_novel = EnrichementAnalysis.EnrichmentAnalysis(y,null_ngrams, {'category':'Novel_Architecture','domain_architecture':'domain_architecture'})
    null_res_list.append(null_enrichment_novel.res.set_index('ngram').True_q)

100%|██████████| 3213/3213 [00:07<00:00, 433.62it/s]


There were 1291 fusion domain architectures previously found in the proteome.


100%|██████████| 3258/3258 [00:06<00:00, 471.14it/s]


There were 1270 fusion domain architectures previously found in the proteome.


100%|██████████| 3192/3192 [00:06<00:00, 465.27it/s]


...

100%|██████████| 3248/3248 [00:06<00:00, 526.43it/s]


There were 1246 fusion domain architectures previously found in the proteome.


100%|██████████| 3272/3272 [00:06<00:00, 514.11it/s]


There were 1278 fusion domain architectures previously found in the proteome.


100%|██████████| 5949/5949 [00:09<00:00, 649.38it/s]


In [9]:
ccle_null_dists = pd.concat(null_res_list, axis = 1)
ccle_null_dists.to_csv('Null_p_val_dists_6-5K_ccle.csv')

In [10]:
ccle_null_dists.loc['IPR000719']

True_q    7.927066e-04
True_q    2.857414e-03
True_q    3.236888e-03
True_q    3.460205e-04
True_q    1.425114e-03
              ...     
True_q    4.274390e-04
True_q    3.481268e-05
True_q    1.803381e-01
True_q    7.766685e-04
True_q    6.046839e-12
Name: IPR000719, Length: 100, dtype: float64